# 4.2. Как провисает нить между опорами


Уравнение нити: $y''=\sqrt{1+(y')^2}/a$. Введём наклон $v=y'$ и длину $\ell$:
$$y'=v,\qquad v'=\frac{\sqrt{1+v^2}}{a},\qquad \ell'=\sqrt{1+v^2}.$$
Начальные условия — $(y,v,\ell)=(0,s,0)$. Для заданных $s$ и $a$ интегрируем систему методом RK4.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq


def rhs(u, a):
    y, v, length = u
    d = np.sqrt(1 + v*v)
    return np.array([v, d/a, d])


def integrate(s, a, n=50):
    x = np.linspace(0, 1, n + 1)
    h = 1/n
    u = np.zeros((n + 1, 3))
    u[0] = [0, s, 0]
    for i in range(n):
        k1 = rhs(u[i], a)
        k2 = rhs(u[i] + h*k1/2, a)
        k3 = rhs(u[i] + h*k2/2, a)
        k4 = rhs(u[i] + h*k3, a)
        u[i+1] = u[i] + h*(k1 + 2*k2 + 2*k3 + k4)/6
    return x, u

## Опоры на одной высоте

При $a=1$ подбираем $s$, чтобы $y(1)=0$. Невязка — значение $y$ на правом конце.
Корень ищем функцией `brentq` на отрезке $[-2,0]$, где невязка меняет знак.
Точный профиль: $y(x)=\cosh(x-1/2)-\cosh(1/2)$.

In [ ]:
def exact(x):
    return np.cosh(x - 0.5) - np.cosh(0.5)


def shoot(n):
    def residual(s):
        return integrate(s, 1, n)[1][-1, 0]
    return brentq(residual, -2, 0, xtol=1e-12)


print("Невязки на концах:", integrate(-2, 1)[1][-1, 0], integrate(0, 1)[1][-1, 0])
s = shoot(50)
x, u = integrate(s, 1)
print(f"Найденный наклон: s = {s:.9f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
for trial in [-1, -0.5, 0]:
    xt, ut = integrate(trial, 1)
    axes[0].plot(xt, ut[:, 0], label=f"s = {trial}")
axes[0].plot([0, 1], [0, 0], "ko", label="опоры")
axes[0].set_title("Пристрелочные профили")
axes[1].plot(x, u[:, 0], label="стрельба")
axes[1].plot(x, exact(x), "--", label="точное решение")
axes[1].plot([0, 1], [0, 0], "ko")
axes[1].set_title("Найденная форма нити")
for ax in axes:
    ax.set(xlabel="x", ylabel="y")
    ax.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for n in [10, 20, 40]:
    s = shoot(n)  # На каждой сетке заново подбираем наклон.
    x, u = integrate(s, 1, n)
    error = np.abs(u[:, 0] - exact(x))
    print(f"h = {1/n:.3f}: |y(1)| = {abs(u[-1, 0]):.2e}, ошибка профиля = {error.max():.2e}")
    ax.plot(x, error, label=f"h = {1/n:.3f}")
ax.set(xlabel="x", ylabel="|y − y точное|", title="Ошибка профиля")
ax.legend()
plt.show()

Невязка на опоре уже мала, но ошибка внутри интервала заметно больше.
При уменьшении шага она убывает. Точный подбор наклона не устраняет ошибку интегрирования.

## Разные высоты и заданная длина

Теперь $y(1)=0.2$, $L=1.2$. Подбираем два числа: $s$ и $b=\log a$, чтобы $a=e^b>0$.
Невязки: $R_1=y(1)-0.2$, $R_2=\ell(1)-1.2$.
В методе Ньютона считаем якобиан центральными разностями и решаем $J\Delta p=-R$.
Останавливаемся, когда обе невязки по модулю меньше $10^{-10}$.

In [ ]:
def residual(p, n=50):
    s, b = p
    x, u = integrate(s, np.exp(b), n)
    return u[-1, [0, 2]] - [0.2, 1.2]


p = np.array([-0.5, np.log(0.6)])
history = []
errors = []
eps = 1e-5

for k in range(20):
    r = residual(p)
    history.append(p.copy())
    errors.append(r)
    if np.max(np.abs(r)) < 1e-10:
        break
    J = np.zeros((2, 2))
    for j in range(2):
        dp = np.zeros(2)
        dp[j] = eps
        J[:, j] = (residual(p + dp) - residual(p - dp))/(2*eps)
    p += np.linalg.solve(J, -r)
else:
    raise RuntimeError("Метод Ньютона не сошёлся за 20 итераций")

s, a = p[0], np.exp(p[1])
print(f"Поправок Ньютона: {k}")
print(f"s = {s:.10f}, a = {a:.10f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
for k, pk in enumerate(history):
    x, u = integrate(pk[0], np.exp(pk[1]))
    axes[0].plot(x, u[:, 0], label=f"k = {k}")
axes[0].plot([0, 1], [0, 0.2], "ko", label="опоры")
axes[0].set(xlabel="x", ylabel="y", title="Форма нити по итерациям Ньютона")
axes[0].legend()

errors = np.abs(np.array(errors))
axes[1].semilogy(range(len(errors)), errors[:, 0], "o-", label="|y(1) − 0.2|")
axes[1].semilogy(range(len(errors)), errors[:, 1], "o-", label="|ℓ(1) − 1.2|")
axes[1].set(xlabel="Итерация k", ylabel="Модуль невязки", title="Сходимость Ньютона")
axes[1].legend()
plt.show()

In [ ]:
for n in [50, 200, 400]:
    x, u = integrate(s, a, n)
    print(f"h = {1/n:.4f}: y(1) = {u[-1, 0]:.10f}, L = {u[-1, 2]:.10f}")
    print(f"  Ошибка высоты: {abs(u[-1, 0] - 0.2):.2e}; ошибка длины: {abs(u[-1, 2] - 1.2):.2e}")